<b><font size="6" color="#E8800A">Week 14a: Support vector machines</font></b><br>
<b><font size="4">Margins, kernels and model selection</font></b><br>

A support vector classifier balances a wide margin against training violations.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Imports</font>](#imports)
* [<font color='#E8800A'>A boundary and its supporting athletes</font>](#margin)
* [<font color='#E8800A'>Kernels: similarity instead of a straight line</font>](#kernels)
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>Assess the reference</font>](#reference)
* [<font color='#E8800A'>Four margin-based classifier families</font>](#family)
* [<font color='#E8800A'>SVC settings</font>](#settings)
* [<font color='#E8800A'>The fixed search budget</font>](#budget)
* [<font color='#E8800A'>Nested cross-validation</font>](#nested)
* [<font color='#E8800A'>The final search and its winner</font>](#search)
* [<font color='#E8800A'>The course board</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Imports</font> <a class="anchor" id="imports"></a>
[Back to TOC](#toc)

In [ ]:
import sys
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.metrics import f1_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.svm import SVC, LinearSVC, NuSVC
from sklearn.linear_model import SGDClassifier
from tqdm.auto import tqdm
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from sklearn.inspection import DecisionBoundaryDisplay
from joblib import parallel_config

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
from course_helpers import Scoreboard, show_log, record, results_before, result_context, PLOT_BLUE, PLOT_ORANGE
from preprocessing import CachedSearchCV, FoldPreprocessor, PreparedEstimator, load_recipe, outer_folds, outer_splitter, prepared_folds
from svm_scoring import ignore_unknown_category_warning, ignore_svc_search_warnings

random_state = 33
np.random.seed(random_state)
sns.set_theme(style='whitegrid')
set_matplotlib_formats('svg')
plt.rcParams['figure.dpi'] = 150
classification_recipe_log = '../../logs/week_04_feature_work_classification_log.json'
ignore_unknown_category_warning()


# <font color='#E8800A'>A boundary and its supporting athletes</font> <a class="anchor" id="margin"></a>
[Back to TOC](#toc)

A linear SVM assigns the score $f(x)=w^\top x+b$. Here $x$ is one row of features, $w$ gives one fitted weight per feature, $w^\top x$ sums their weighted values, and $b$ is an intercept. A positive score predicts the second class in `classes_`; a negative score predicts the first. A score is not a probability.

The two margin lines have scores $-1$ and $+1$. Their geometric separation is $2/\lVert w\rVert$, where $\lVert w\rVert$ is the length of the weight vector. Soft-margin training minimises

$$\frac12\lVert w\rVert^2+C\sum_{i=1}^{n}\max(0,1-y_i f(x_i)).$$

The index $i$ identifies one of the $n$ training rows, and $y_i$ is its class written as $-1$ or $+1$. The maximum is hinge loss: zero outside the correct margin and positive for a margin violation. A point inside the margin can still be classified correctly. `C` weights violations against the squared-weight penalty. Increasing it makes violations more costly; it does not guarantee better predictions.

Support vectors have nonzero coefficients in the fitted decision function. In a soft-margin fit they may lie on the margin, inside it, or on the wrong side. Rows with zero coefficients make no contribution to this fitted boundary; changing data and refitting can change which rows become support vectors. The examples below use two artificial features so their margins can be drawn without projecting the athlete data.

<div class="alert alert-block alert-info">

**`SVC(C=1.0, kernel="rbf", gamma="scale")`** fits a support vector classifier.

**Main Parameters:**
- `C`: positive violation penalty; larger values reduce regularisation.
- `kernel`: `linear`, `rbf`, `poly` or `sigmoid` similarity rule.
- `gamma`: kernel coefficient for the nonlinear kernels; ignored by linear.
- `class_weight`: `None` or `balanced`, which weights each class inversely to its fitting frequency.
- `max_iter`: solver iteration cap; `-1` removes the cap.
- `probability=False`: avoids the additional internal calibration used to enable `predict_proba`.

**Main Attributes:** `support_` identifies supporting fitting rows; `support_vectors_` contains their prepared features; `n_support_` counts them by class; `dual_coef_` and `intercept_` define the kernel decision function. `coef_` exists for a linear kernel only. `fit_status_=0` reports successful convergence.

**Main Methods:** `fit(X, y)`, `predict(X)` and `decision_function(X)`. `score(X, y)` reports accuracy, so this notebook calculates F1 explicitly. Nonlinear decision scores are not geometric distances in the original feature units.

</div>

### Separable classes and the street between them

Two artificial data sets, each with features X1 and X2, show how the street between the classes behaves. The first is separable, and a very large C makes margin violations so costly that the fit approximates a hard margin. The score $f(x)=w^\top x+b$ equals $-1$ on the lower dashed line, 0 on the solid boundary and $+1$ on the upper dashed line, and the street between the dashed lines has width $2/\lVert w\rVert$.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.inspection.DecisionBoundaryDisplay.html'>sklearn.inspection.DecisionBoundaryDisplay.from_estimator(estimator, X, response_method="auto", plot_method="contourf", eps=1.0, grid_resolution=100, ax=None)</a>

**Definition:**
Evaluate a fitted estimator that has two features on a grid and draw the response.

**Main Parameters:**
- `estimator`, `X`: the fitted model and the two-column data that sets the plotted range
- `response_method`: `predict` draws classes, `decision_function` draws scores
- `plot_method`: `contour`, `contourf` or `pcolormesh`
- `eps`: margin added around the data range; `grid_resolution`: grid points per axis
- `levels`, `colors`, `linestyles`, `cmap`: passed to the matplotlib plot method

**Main Attributes (after drawing):** `xx0`, `xx1` (the grid), `response` (the grid values), `surface_` (the drawn surface), `ax_` and `figure_`.
</div>

__Step 1:__ Define the two class colours.

In [ ]:
toy_class_colors = ListedColormap([PLOT_BLUE, PLOT_ORANGE])
toy_score_colors = LinearSegmentedColormap.from_list("toy_score", [PLOT_BLUE, "white", PLOT_ORANGE])

__Step 2:__ Define and inspect the separable two-feature data.

In [ ]:
toy_sep_rows = np.array([[3, 7, 0], [12, 6, 0], [18, 6, 0], [12, 6, 0], [3, 7, 0], [7, 5, 0],
                         [10, 10, 0], [17, 8, 0], [17, 11, 0], [26, 19, 1], [20, 20, 1], [17, 19, 1],
                         [23, 15, 1], [25, 9, 1], [21, 14, 1], [31, 8, 1], [24, 12, 1], [23, 11, 1]])
toy_sep_X = toy_sep_rows[:, :2]
toy_sep_y = toy_sep_rows[:, 2]
display(pd.DataFrame(toy_sep_rows, columns=["x1", "x2", "class"]))


__Step 3:__ Fit the linear SVC after inspecting the separable rows.

In [ ]:
toy_sep_model = SVC(kernel="linear", C=1000).fit(toy_sep_X, toy_sep_y)


__Step 4:__ Inspect weights, support rows, street width and training errors.

In [ ]:
toy_sep_w = toy_sep_model.coef_[0]
toy_sep_b = toy_sep_model.intercept_[0]
toy_sep_norm = np.linalg.norm(toy_sep_w)
print("w:", toy_sep_w.round(4), "b:", round(toy_sep_b, 4))
print("support rows:", toy_sep_model.support_, "at", toy_sep_X[toy_sep_model.support_].tolist())
print(f"street width 2/||w||: {2 / toy_sep_norm:.4f}")
print("training errors:", int((toy_sep_model.predict(toy_sep_X) != toy_sep_y).sum()))

Three support vectors fix the fit: rows 8, 11 and 17, at (17, 11), (17, 19) and (23, 11). The weights are 0.3333 and 0.25 with intercept -9.4167, so the street between the dashed lines is 4.80 wide and no training row is misclassified.

__Step 5:__ Draw the boundary, the margins and the weight vector.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
DecisionBoundaryDisplay.from_estimator(
    toy_sep_model, toy_sep_X, response_method="decision_function", plot_method="contour", eps=4,
    levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"], ax=ax)
for toy_sep_class, toy_sep_color in [(0, PLOT_BLUE), (1, PLOT_ORANGE)]:
    ax.scatter(*toy_sep_X[toy_sep_y == toy_sep_class].T, color=toy_sep_color, s=45, label=f"class {toy_sep_class}")
ax.scatter(*toy_sep_X[toy_sep_model.support_].T, s=190, facecolors="none", edgecolors="k", label="support vectors")
toy_sep_unit = toy_sep_w / toy_sep_norm
toy_sep_centre = toy_sep_X.mean(axis=0)
toy_sep_foot = toy_sep_centre - (toy_sep_centre @ toy_sep_w + toy_sep_b) / toy_sep_norm ** 2 * toy_sep_w
toy_sep_along = np.array([-toy_sep_unit[1], toy_sep_unit[0]])
toy_sep_low = toy_sep_foot - 10 * toy_sep_along
ax.annotate("", xy=toy_sep_low + 1 / toy_sep_norm * toy_sep_unit, xytext=toy_sep_low - 1 / toy_sep_norm * toy_sep_unit,
            arrowprops=dict(arrowstyle="<->", color="k", lw=1.5))
ax.text(*(toy_sep_low + 1.4 / toy_sep_norm * toy_sep_unit), f"width 2/||w|| = {2 / toy_sep_norm:.2f}", ha="left", va="center")
toy_sep_start = toy_sep_foot + 9 * toy_sep_along
ax.annotate("", xy=toy_sep_start + 4 * toy_sep_unit, xytext=toy_sep_start,
            arrowprops=dict(arrowstyle="-|>", color="k", lw=2))
ax.text(*(toy_sep_start + 5 * toy_sep_unit), "w", fontsize=13, ha="center", va="center")
ax.set_aspect("equal")
ax.set(xlabel="X1", ylabel="X2", title="Linear SVC with C=1000: boundary, margins and weight vector")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
fig.tight_layout()
plt.show()

The weight vector is perpendicular to the boundary and points toward class 1. The three circled rows sit on the dashed lines, so they alone fix the street; the other fifteen rows do not constrain it.

__Step 6:__ Predict the new point with the fitted linear SVC.

In [ ]:
toy_sep_u = np.array([22.0, 14.0])
print(f"decision score: {toy_sep_model.decision_function([toy_sep_u])[0]:.4f}")
print("predicted class:", toy_sep_model.predict([toy_sep_u])[0])


For $u=(22,14)$, `decision_function` returns approximately 1.4167. Its positive sign gives class 1.

Now add one class-0 athlete at (17, 17), next to class-1 rows. A separating line still exists, but only a narrow street fits between the classes, and C decides whether the fit pays for that narrowness or accepts violations. A margin violation is a row with $y_if(x_i)<1$, where $y_i$ is written as $-1$ or $+1$; counting it with a tolerance of 0.01, the check is $y_if(x_i)<0.99$. The hinge sum $\sum_i\max(0,1-y_if(x_i))$ is the penalty term that C multiplies in the objective.

__Step 7:__ Add a class-0 point near the other class and inspect its coordinates.

In [ ]:
toy_over_X = np.vstack([toy_sep_X, [[17, 17]]])
toy_over_y = np.append(toy_sep_y, 0)
toy_over_signed = 2 * toy_over_y - 1
print("added class-0 input:", toy_over_X[-1])


__Step 8:__ Fit the three C values and inspect margin violations.

In [ ]:
toy_over_models, toy_over_rows = [], []
for toy_over_c in [0.05, 1, 1000]:
    toy_over_fit = SVC(kernel="linear", C=toy_over_c).fit(toy_over_X, toy_over_y)
    toy_over_margin = toy_over_signed * toy_over_fit.decision_function(toy_over_X)
    toy_over_models.append(toy_over_fit)
    toy_over_rows.append({"C": toy_over_c, "margin width": 2 / np.linalg.norm(toy_over_fit.coef_[0]),
                          "support vectors": int(toy_over_fit.n_support_.sum()),
                          "margin violations": int((toy_over_margin < 0.99).sum()),
                          "training errors": int((toy_over_margin < 0).sum()),
                          "hinge sum": np.maximum(0, 1 - toy_over_margin).sum()})
display(pd.DataFrame(toy_over_rows).round(3))

At C=0.05 the street is 6.556 wide, five rows support it, three rows violate the margin, one is misclassified and the hinge sum is 2.138. At C=1000 the street shrinks to 1.200, no row violates the margin and the hinge sum is 0.001. C=1 sits between them, with width 1.667, one violation and hinge sum 0.560. A larger C lowers the hinge sum at the price of a narrower street.

__Step 9:__ Draw the three fits side by side.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharex=True, sharey=True, layout="constrained")
for ax, toy_over_fit, toy_over_row in zip(axes, toy_over_models, toy_over_rows):
    toy_over_display = DecisionBoundaryDisplay.from_estimator(
        toy_over_fit, toy_over_X, response_method="decision_function", plot_method="pcolormesh", eps=4,
        cmap=toy_score_colors, vmin=-3, vmax=3, alpha=0.6, ax=ax)
    ax.grid(False)
    ax.contour(toy_over_display.xx0, toy_over_display.xx1, toy_over_display.response,
               levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"])
    for toy_over_class, toy_over_color in [(0, PLOT_BLUE), (1, PLOT_ORANGE)]:
        ax.scatter(*toy_over_X[toy_over_y == toy_over_class].T, color=toy_over_color, s=40,
                   edgecolors="white", label=f"class {toy_over_class}")
    ax.scatter(*toy_over_fit.support_vectors_.T, s=170, facecolors="none", edgecolors="k", label="support vectors")
    ax.scatter(17, 17, marker="*", s=260, color=PLOT_BLUE, edgecolors="k", label="added class-0 point")
    ax.set(xlabel="X1", title=f"C={toy_over_row['C']}: width {toy_over_row['margin width']:.2f}, "
                              f"{toy_over_row['support vectors']} support vectors")
axes[0].set_ylabel("X2")
fig.colorbar(toy_over_display.surface_, ax=axes, label="decision score f(x)", shrink=0.85)
fig.legend(*axes[-1].get_legend_handles_labels(), loc="outside lower center", ncol=4, frameon=False)
plt.show()

The shaded score changes slowly across the wide street at C=0.05 and sharply across the narrow one at C=1000. The three and one violations counted above are the rows inside the street in the left and middle panels. The star marks the added class-0 athlete.

# <font color='#E8800A'>Kernels: similarity instead of a straight line</font> <a class="anchor" id="kernels"></a>
[Back to TOC](#toc)

A straight boundary cannot separate every data set. The sixteen artificial points below, with two features $x_1$ and $x_2$, come from the scikit-learn documentation of kernel boundaries.

__Step 10:__ Define the sixteen points and inspect their class counts.

In [ ]:
kernel_toy_X = np.array([[0.4, -0.7], [-1.5, -1.0], [-1.4, -0.9], [-1.3, -1.2], [-1.1, -0.2], [-1.2, -0.4],
                         [-0.5, 1.2], [-1.5, 2.1], [1.0, 1.0], [1.3, 0.8], [1.2, 0.5], [0.2, -2.0],
                         [0.5, -2.4], [0.2, -2.3], [0.0, -2.7], [1.3, 2.1]])
kernel_toy_y = np.array([0] * 8 + [1] * 8)
print("rows per class:", np.bincount(kernel_toy_y))


__Step 11:__ Fit a linear SVC to the sixteen defined points.

In [ ]:
kernel_toy_linear = SVC(kernel="linear").fit(kernel_toy_X, kernel_toy_y)
print("rows per class:", np.bincount(kernel_toy_y))
print("linear SVC training accuracy:", kernel_toy_linear.score(kernel_toy_X, kernel_toy_y))

The classes hold 8 rows each, and the linear SVC classifies 0.9375 of them correctly, which is 15 of 16.

#### The kernel trick

A positive-semidefinite kernel replaces the dot product between two rows by a similarity $K(x,x')=\phi(x)^\top\phi(x')$, where $\phi$ maps a row into a feature space of higher dimension and $K$ returns the inner product there without building $\phi$. The supported sigmoid option does not guarantee this positive-semidefinite property. Training and prediction use the rows only through $K$, so the decision function is

$$f(x)=\sum_{i\in SV}y_i\alpha_iK(x_i,x)+b.$$

Here $SV$ indexes the support vectors, $y_i\in\{-1,+1\}$ is the class of support vector $i$, $\alpha_i$ is its fitted dual weight with $0\le\alpha_i\le C$, and $b$ is `intercept_`. The attribute `dual_coef_` stores the products $y_i\alpha_i$. The degree-2 polynomial kernel with $\gamma=1$ and $r=0$ shows the idea, because its map can be written out: for $x=(x_1,x_2)$, $\phi(x)=(x_1^2,\sqrt2\,x_1x_2,x_2^2)$.

__Step 12:__ Define signed inputs with class 0 in the centre and class 1 in both tails.

In [ ]:
map_X = np.array([-2.8, -2.2, -1.8, -0.8, -0.4, 0.0, 0.4, 0.8, 1.8, 2.2, 2.8]).reshape(-1, 1)
map_y = (np.abs(map_X[:, 0]) > 1).astype(int)
display(pd.DataFrame({"x": map_X[:, 0], "phi(x) = x squared": map_X[:, 0] ** 2, "class": map_y}))


The two tails share class 1, while the centre has class 0. No single threshold on signed x separates them. Squaring maps both tails to large positive values, so one threshold on $\phi(x)=x^2$ can separate the classes.

__Step 13:__ Fit one homogeneous quadratic-kernel SVC and inspect its class order.

In [ ]:
map_model = SVC(kernel="poly", degree=2, gamma=1, coef0=0, C=10).fit(map_X, map_y)
print(map_model)
print("class order:", map_model.classes_)
print("support inputs:", map_model.support_vectors_[:, 0])


__Step 14:__ Evaluate the fitted model once on a signed grid and locate the zero-score threshold.

In [ ]:
map_grid = np.linspace(-3, 3, 601).reshape(-1, 1)
map_scores = map_model.decision_function(map_grid)
map_predictions = map_model.predict(map_grid)
map_phi = map_grid[:, 0] ** 2
map_order = np.argsort(map_phi)
map_threshold = float(np.interp(0, map_scores[map_order], map_phi[map_order]))
print(f"phi threshold: {map_threshold:.3f}; x thresholds: {-np.sqrt(map_threshold):.3f}, {np.sqrt(map_threshold):.3f}")
print("grid predicted class counts:", np.bincount(map_predictions))


__Step 15:__ Plot those same scores against the mapped feature and the original input.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
axes[0].plot(map_phi[map_order], map_scores[map_order], color="black", label="same fitted decision score")
axes[1].plot(map_grid[:, 0], map_scores, color="black", label="same fitted decision score")
for label, color in [(0, PLOT_BLUE), (1, PLOT_ORANGE)]:
    mask = map_y == label
    # Class markers sit on a labelled strip below the score curve.
    axes[0].scatter(map_X[mask, 0] ** 2, np.full(mask.sum(), -3.0), color=color, s=45, label=f"training class {label}")
    axes[1].scatter(map_X[mask, 0], np.full(mask.sum(), -3.0), color=color, s=45, label=f"training class {label}")
for ax in axes:
    ax.axhline(0, color="0.4", linestyle="--", label="zero score: class threshold")
    ax.set(ylabel="decision score f(x)")
    ax.text(0.5, 0.96, "f > 0: class 1 (tails)\nf < 0: class 0 (centre)", transform=ax.transAxes, va="top", ha="center")
axes[0].axvline(map_threshold, color="0.5", linestyle=":")
for threshold in [-np.sqrt(map_threshold), np.sqrt(map_threshold)]:
    axes[1].axvline(threshold, color="0.5", linestyle=":")
axes[0].set(xlabel="phi(x) = x squared", title="One linear score in the mapped feature")
axes[1].set(xlabel="signed input x", title="The same score bends in the original input")
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=4, frameon=False, fontsize=9)
fig.tight_layout(rect=(0, 0.12, 1, 1))
plt.show()


With $K(x,xÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â²)=(xxÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â²)^2$, the fitted score is linear in $\phi(x)=x^2$ and quadratic in x. The left panel sorts the mapped coordinates before joining them. Its one zero crossing becomes two symmetric thresholds on the right. Both panels use the same fitted classifier and the same grid scores; changing coordinates does not refit the model.

#### The four kernels

Each kernel compares two rows $x$ and $x'$ with $p$ features. Two quantities carry the geometry: the dot product $\langle x,x'\rangle=\sum_{j=1}^{p}x_jx'_j=\lVert x\rVert\lVert x'\rVert\cos\theta$, where $\theta$ is the angle between the rows seen as vectors from the origin, and the squared distance $\lVert x-x'\rVert^2=\sum_{j=1}^{p}(x_j-x'_j)^2$.

| Kernel | Formula | Symbols | What it measures |
|---|---|---|---|
| `linear` | $\langle x,x'\rangle$ | none | the dot product alone |
| `poly` | $(\gamma\langle x,x'\rangle+r)^d$ | $\gamma>0$ is `gamma`, $r$ is `coef0`, $d$ is `degree` | the dot product, shifted by $r$ and raised to the power $d$ |
| `rbf` | $\exp(-\gamma\lVert x-x'\rVert^2)$ | $\gamma>0$ is `gamma` | the distance between the rows, and nothing else |
| `sigmoid` | $\tanh(\gamma\langle x,x'\rangle+r)$ | $\gamma$ is `gamma`, $r$ is `coef0` | the dot product, shifted by $r$ and squashed into $(-1,1)$ |

__Step 16:__ Draw the RBF similarity against distance for four values of gamma.

In [ ]:
kernel_distance = np.linspace(0, 3, 301)
kernel_gammas = [0.1, 0.5, 2.0, 10.0]
for kernel_g in kernel_gammas:
    print(f"gamma={kernel_g}: similarity falls to 0.5 at distance {np.sqrt(np.log(2) / kernel_g):.3f}")


__Step 17:__ Plot the RBF similarities using the declared distances and gamma values.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for kernel_g, kernel_shade in zip(kernel_gammas, plt.cm.Blues(np.linspace(0.4, 1.0, len(kernel_gammas)))):
    ax.plot(kernel_distance, rbf_kernel(kernel_distance.reshape(-1, 1), [[0.0]], gamma=kernel_g)[:, 0], color=kernel_shade, linewidth=2.2, label=f"gamma={kernel_g}")
ax.axhline(0.5, color="0.6", linestyle=":")
ax.set(xlabel="distance ||x - x'||", ylabel="RBF similarity K", title="RBF similarity against distance for four gamma values")
ax.legend()
fig.tight_layout()
plt.show()

The RBF similarity falls to 0.5 at distance 2.633 for $\gamma=0.1$ and at 0.263 for $\gamma=10$. A larger gamma shrinks the reach of one support vector, so only rows close to it are affected.

__Step 18:__ Fit one SVC per kernel on the sixteen points.

In [ ]:
kernel_gamma, kernel_coef0, kernel_degree = 2.0, 0.0, 3
kernel_models = {kernel_name: SVC(kernel=kernel_name, gamma=kernel_gamma, coef0=kernel_coef0, degree=kernel_degree).fit(kernel_toy_X, kernel_toy_y)
                 for kernel_name in ["linear", "poly", "rbf", "sigmoid"]}


__Step 19:__ Inspect training accuracy and support counts for each fitted kernel.

In [ ]:
display(pd.DataFrame([{"kernel": kernel_name, "training accuracy": kernel_fit.score(kernel_toy_X, kernel_toy_y),
                       "support vectors": int(kernel_fit.n_support_.sum())}
                      for kernel_name, kernel_fit in kernel_models.items()]))

The linear SVC classifies 0.9375 of the rows with 5 support vectors, poly and rbf classify all of them with 4 and 12 support vectors, and sigmoid classifies 0.5625 with 12.

__Step 20:__ Draw the four decision boundaries.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, (kernel_name, kernel_fit) in zip(axes.ravel(), kernel_models.items()):
    ax.grid(False)
    DecisionBoundaryDisplay.from_estimator(kernel_fit, kernel_toy_X, response_method="predict", plot_method="pcolormesh",
                                           cmap=toy_class_colors, alpha=0.25, eps=1.7, grid_resolution=300, ax=ax, xlabel="x1", ylabel="x2")
    DecisionBoundaryDisplay.from_estimator(kernel_fit, kernel_toy_X, response_method="decision_function", plot_method="contour",
                                           levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"], eps=1.7, grid_resolution=300, ax=ax)
    for kernel_class, kernel_color in [(0, PLOT_BLUE), (1, PLOT_ORANGE)]:
        ax.scatter(*kernel_toy_X[kernel_toy_y == kernel_class].T, color=kernel_color, s=40, edgecolors="k", label=f"class {kernel_class}")
    ax.scatter(*kernel_fit.support_vectors_.T, s=170, facecolors="none", edgecolors="k", label="support vectors")
    ax.set(xlim=(-3, 3), ylim=(-3, 3), title=f"{kernel_name}: {kernel_fit.score(kernel_toy_X, kernel_toy_y):.3f} accuracy, "
                                              f"{int(kernel_fit.n_support_.sum())} support vectors")
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

The linear boundary is straight and leaves one row wrong. The poly boundary bends, and the rbf contours close around the groups of rows. The sigmoid regions are irregular and give the lowest training accuracy of the four. These are training accuracies on sixteen artificial rows, so they illustrate the shapes rather than rank the kernels.

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

__Step 21:__ Load the data, set aside the test rows and prepare the training folds.

In [ ]:
champions, classification_recipe = load_recipe(
    "../../data/interim/champions.csv", classification_recipe_log)
champions = champions.set_index("RecordID")
X = champions[classification_recipe["categorical"] + classification_recipe["numeric"]]
y = champions["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=random_state)

# cache_size=256 MB is the per-fit kernel cache, which reuses kernel values.
fixed_settings = {"random_state": random_state, "cache_size": 256}
svm_prep = FoldPreprocessor(classification_recipe).fit(X_train, y_train)
Z_train = svm_prep.transform(X_train)
Z_test = svm_prep.transform(X_test)
folds = outer_splitter("classification", random_state=random_state)
prepared = prepared_folds(FoldPreprocessor(classification_recipe), folds, X_train, y_train)
svm_log = []
print(f"{len(y_train):,} training athletes; {len(y_test):,} held-out athletes")


# <font color='#E8800A'>Assess the reference</font> <a class="anchor" id="reference"></a>
[Back to TOC](#toc)

The reference is `SVC(C=1, kernel="rbf", gamma="scale")`, with no iteration cap. The ten prepared folds assess it before any setting changes.

__Step 22:__ Measure the reference on the prepared training folds.

In [ ]:
reference_train, reference_validation = [], []
for fold in prepared:
    reference_fold = SVC(**fixed_settings).fit(fold["X_fit"], fold["y_fit"])
    reference_train.append(f1_score(fold["y_fit"], reference_fold.predict(fold["X_fit"])))
    reference_validation.append(f1_score(fold["y_score"], reference_fold.predict(fold["X_score"])))
default_scores = {"train_score": np.array(reference_train), "test_score": np.array(reference_validation)}
default_estimator = PreparedEstimator(FoldPreprocessor(classification_recipe), SVC(**fixed_settings))


__Step 23:__ Fit the reference on all training athletes.

In [ ]:
fitted_reference = clone(default_estimator).fit(X_train, y_train)
reference = {"configuration": " ".join(repr(fitted_reference.estimator_).split()),
    "train": default_scores["train_score"], "validation": default_scores["test_score"],
    "model": fitted_reference}


The fitted decision score is $f(x)=\sum_{j\in SV}a_jK(s_j,x)+b$. Here $s_j$ is a support vector, $a_j$ is its signed dual coefficient and $b$ is the intercept. The model applies preprocessing and evaluates this score through `decision_function`; `predict` returns the class.

__Step 24:__ Inspect the default representation and solver status.

In [ ]:
default_svm = fitted_reference.estimator_
print("Support-vector matrix:", default_svm.support_vectors_.shape)
print("Support vectors by class:", default_svm.n_support_)
print("Solver status:", default_svm.fit_status_)


The fitted default stores 1581 support vectors with 28 prepared features each: 750 from class 0 and 831 from class 1. Solver status 0 reports successful optimisation. Support counts describe the fitted representation rather than prediction accuracy.

__Step 25:__ Read one held-out decision score and predicted class.

In [ ]:
default_query = X_test.iloc[:1]
print("Decision score:", fitted_reference.decision_function(default_query)[0])
print("Actual athlete:", y_test.iloc[0], "prediction:", fitted_reference.predict(default_query)[0])


The decision score is 0.85375. Its positive sign gives class 1, matching this athlete.

__Step 26:__ Record the reference and its fold scores.

In [ ]:
record(svm_log, {"frame": type(folds).__name__, "rows": len(y_train),
                 "folds": folds.get_n_splits(), "name": reference["configuration"],
                 "train": reference["train"], "validation": reference["validation"], "model": reference["model"]},
       X_test, y_test, "untuned", metric=f1_score, label="F1")

# <font color='#E8800A'>Four margin-based classifier families</font> <a class="anchor" id="family"></a>
[Back to TOC](#toc)

The same athlete folds let us compare four ways of fitting a margin classifier. All four expose `fit`, `predict`, `decision_function` and `classes_`. Only the kernel estimators retain support vectors; the two linear estimators expose one weight per prepared feature through `coef_` and an intercept through `intercept_`.

| Family | Boundary and objective | Regularisation and optimisation | Useful fitted attributes |
|---|---|---|---|
| `SVC` | Linear or kernel boundary; hinge-loss margin objective | `C` penalises violations; LIBSVM solves the dual problem | `support_`, `support_vectors_`, `dual_coef_`, `n_support_`, `fit_status_` |
| `LinearSVC` | Linear boundary; default squared hinge loss penalises large violations more strongly than hinge loss | Default L2 penalty; `C` is inverse regularisation strength; LIBLINEAR optimises a linear problem, with an intercept penalty | `coef_`, `intercept_`, `n_iter_` |
| `NuSVC` | Linear or kernel boundary; nu formulation sets a margin/error constraint instead of the SVC C trade-off | `nu` bounds training margin errors above and the support-vector fraction below; LIBSVM solves the dual problem | `support_`, `support_vectors_`, `dual_coef_`, `n_support_`, `fit_status_` |
| `SGDClassifier(loss="hinge")` | Linear boundary with hinge loss | `alpha` weights the penalty rather than using `C`; stochastic gradient updates depend on the learning schedule and stopping rule | `coef_`, `intercept_`, `n_iter_`, `t_` |

The linear estimators avoid a kernel matrix, which can help on larger data. Their objectives, intercept treatment and optimisation differ, so matching a numeric C or alpha does not make their fits equivalent. SGD has no `support_vectors_` attribute, even though its hinge-loss objective is related to a linear SVM. A fixed seed makes its row order reproducible.

For binary NuSVC, each fitting partition must satisfy $\nu\le 2\min(n_0,n_1)/(n_0+n_1)$. The joint search below checks its nu values against every inner fitting partition before any candidate is scored.

__Step 27:__ Compare the four reference families on the shared athlete folds and record their fitted models.

In [ ]:
family_references = {
    "SVC": SVC(**fixed_settings),
    "LinearSVC": LinearSVC(random_state=random_state, dual=True, max_iter=10000),
    "NuSVC": NuSVC(nu=0.25, random_state=random_state, cache_size=256),
    "SGDClassifier": SGDClassifier(loss="hinge", random_state=random_state, max_iter=5000),
}
family_rows = [{"family": "SVC", "train F1": np.mean(reference["train"]),
                "cv F1": np.mean(reference["validation"])}]
for family, estimator in family_references.items():
    if family == "SVC":
        continue
    train_scores, validation_scores = [], []
    warning_count = 0
    for fold in prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted_family = clone(estimator).fit(fold["X_fit"], fold["y_fit"])
        warning_count += sum(issubclass(w.category, ConvergenceWarning) for w in caught)
        train_scores.append(f1_score(fold["y_fit"], fitted_family.predict(fold["X_fit"])))
        validation_scores.append(f1_score(fold["y_score"], fitted_family.predict(fold["X_score"])))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        fitted_family = PreparedEstimator(FoldPreprocessor(classification_recipe), clone(estimator)).fit(X_train, y_train)
    warning_count += sum(issubclass(w.category, ConvergenceWarning) for w in caught)
    record(svm_log, {"frame": type(folds).__name__, "rows": len(y_train),
        "folds": folds.get_n_splits(), "name": " ".join(repr(estimator).split()),
        "train": train_scores, "validation": validation_scores, "model": fitted_family},
        X_test, y_test, "untuned family reference", metric=f1_score, label="F1")
    family_rows.append({"family": family, "train F1": np.mean(train_scores),
        "cv F1": np.mean(validation_scores), "convergence warnings": warning_count})
family_table = pd.DataFrame(family_rows).fillna({"convergence warnings": 0})
display(family_table)


The SVC reference has the highest mean validation F1, 0.8587, followed by LinearSVC at 0.8510, NuSVC at 0.8356 and hinge-loss SGD at 0.8332. NuSVC has the closest training fit, 0.9607, but the largest training-validation gap. These fixed references compare starting configurations; the joint search below selects each family together with its settings.

# <font color='#E8800A'>SVC settings</font> <a class="anchor" id="settings"></a>
[Back to TOC](#toc)

Each comparison changes one SVC argument on the same prepared folds. The log records the row with the highest mean validation F1. Test F1 describes the fitted rows.

| Setting | Compared options |
|---|---|
| `C` | 0.01, 0.1, 1, 10, 100 |
| `gamma` | scale, 0.001, 0.01, 0.1, 1, 10 |
| `kernel` | linear, rbf, poly, sigmoid |
| `max_iter` | 100, 1000, 10000, -1 |
| `class_weight` | None, balanced |

### `SVC: C`

Small C tolerates more hinge loss to favour a wider margin. Large C makes individual violations more costly. Read the training-validation gap rather than assuming that a closer training fit predicts better.

__Step 28:__ Compare `C` while keeping the other defaults. Fill the value list with `[0.01, 0.1, 1.0, 10.0, 100.0]` and measure every row on the shared folds.

In [ ]:
c_values = ...

__Step 29:__ Measure every c value on the prepared folds and fit each configuration on all training rows.

In [ ]:
c_rows = []
for value in tqdm(c_values, desc='C', leave=True, position=0):
    settings = {**fixed_settings, "C": value}
    train_scores, validation_scores, warning_counts = [], [], []
    for fold in prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = SVC(**settings).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
        warning_counts.append(sum(issubclass(w.category, ConvergenceWarning) for w in caught))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        fitted = SVC(**settings).fit(Z_train, y_train)
    c_rows.append({"name": f"C={value}", "settings": settings,
        "train": train_scores, "validation": validation_scores, "model": fitted,
        "configuration": " ".join(repr(SVC(**settings)).split()),
        "train F1": np.mean(train_scores), "cv F1": np.mean(validation_scores),
        "support vectors": int(fitted.n_support_.sum()), "iterations": int(fitted.n_iter_.max()),
        "fit status": int(fitted.fit_status_), "fold warnings": sum(warning_counts)})
c_table = pd.DataFrame(c_rows)

__Step 30:__ Display the c comparison with its held-out F1 and solver diagnostics.

In [ ]:
c_table["test F1"] = [f1_score(y_test, row["model"].predict(Z_test)) for row in c_rows]
display(c_table.drop(columns=["settings", "train", "validation", "model"]).round(4))


C=1 gives the largest mean validation F1, 0.8587; C=100 fits the training rows more closely but has lower validation F1, 0.8217.

__Step 31:__ Record the validation-selected configuration.

In [ ]:
c_best = c_rows[int(c_table["cv F1"].idxmax())]
svm_log[:] = [row for row in svm_log if row["validation F1 is"] != 'one setting changed: C']
record(svm_log, {"frame": type(folds).__name__, "rows": len(y_train),
                 "folds": folds.get_n_splits(), "name": c_best["configuration"],
                 "train": c_best["train"], "validation": c_best["validation"],
                 "model": c_best["model"]},
       Z_test, y_test, 'one setting changed: C', metric=f1_score, label="F1")


__Step 32:__ Plot the measured comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
positions = np.arange(len(c_table))
for column, marker in [("train F1", "o"), ("cv F1", "s"), ("test F1", "^")]:
    ax.plot(positions, c_table[column], marker=marker, label=column)
ax.set(xticks=positions, xticklabels=c_table["name"].str.split("=").str[-1], xlabel='C', ylabel="F1",
       title="SVC: C compared on the shared folds")
ax.legend()
fig.tight_layout()
plt.show()


The largest validation F1 in this table is 0.8587, from `C=1.0`; its training F1 is 0.8874 and its held-out F1 is 0.8649. Increasing C from 0.01 to 100 changes training F1 from 0.7730 to 0.9781, while validation F1 changes from 0.7715 to 0.8217. The validation scores show whether paying more for training violations helps excluded athletes.

### `SVC: gamma`

For RBF, $K(x,z)=\exp(-\gamma\lVert x-z\rVert^2)$. The vectors $x$ and $z$ are two prepared rows, and the squared norm is their squared Euclidean distance. A larger positive $\gamma$ makes similarity fall faster with distance, giving each row a more local influence. `scale` sets gamma to $1/(p\,\mathrm{Var}(X))$, where $p$ is the number of prepared features and $\mathrm{Var}(X)$ is the variance of all entries of the fitting matrix.

__Step 33:__ Compare `gamma` while keeping the other defaults. Fill the value list with `["scale", 0.001, 0.01, 0.1, 1.0, 10.0]` and measure every row on the shared folds.

In [ ]:
gamma_values = ...

__Step 34:__ Measure every gamma value on the prepared folds and fit each configuration on all training rows.

In [ ]:
gamma_rows = []
for value in tqdm(gamma_values, desc='gamma', leave=True, position=0):
    settings = {**fixed_settings, "gamma": value}
    train_scores, validation_scores, warning_counts = [], [], []
    for fold in prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = SVC(**settings).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
        warning_counts.append(sum(issubclass(w.category, ConvergenceWarning) for w in caught))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        fitted = SVC(**settings).fit(Z_train, y_train)
    gamma_rows.append({"name": f"gamma={value}", "settings": settings,
        "train": train_scores, "validation": validation_scores, "model": fitted,
        "configuration": " ".join(repr(SVC(**settings)).split()),
        "train F1": np.mean(train_scores), "cv F1": np.mean(validation_scores),
        "support vectors": int(fitted.n_support_.sum()), "iterations": int(fitted.n_iter_.max()),
        "fit status": int(fitted.fit_status_), "fold warnings": sum(warning_counts)})
gamma_table = pd.DataFrame(gamma_rows)

__Step 35:__ Display the gamma comparison with its held-out F1 and solver diagnostics.

In [ ]:
gamma_table["test F1"] = [f1_score(y_test, row["model"].predict(Z_test)) for row in gamma_rows]
display(gamma_table.drop(columns=["settings", "train", "validation", "model"]).round(4))


The scale row has the largest validation F1, 0.8587. Gamma=10 almost perfectly fits training athletes, while validation F1 falls to 0.7529.

__Step 36:__ Record the validation-selected configuration.

In [ ]:
gamma_best = gamma_rows[int(gamma_table["cv F1"].idxmax())]
svm_log[:] = [row for row in svm_log if row["validation F1 is"] != 'one setting changed: gamma']
record(svm_log, {"frame": type(folds).__name__, "rows": len(y_train),
                 "folds": folds.get_n_splits(), "name": gamma_best["configuration"],
                 "train": gamma_best["train"], "validation": gamma_best["validation"],
                 "model": gamma_best["model"]},
       Z_test, y_test, 'one setting changed: gamma', metric=f1_score, label="F1")


__Step 37:__ Plot the measured comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
positions = np.arange(len(gamma_table))
for column, marker in [("train F1", "o"), ("cv F1", "s"), ("test F1", "^")]:
    ax.plot(positions, gamma_table[column], marker=marker, label=column)
ax.set(xticks=positions, xticklabels=gamma_table["name"].str.split("=").str[-1], xlabel='gamma', ylabel="F1",
       title="SVC: gamma compared on the shared folds")
ax.legend()
fig.tight_layout()
plt.show()


The largest validation F1 in this table is 0.8587, from `gamma=scale`; its training F1 is 0.8874 and its held-out F1 is 0.8649. At gamma=10, training F1 is 0.9987 and validation F1 is 0.7529, a gap of 0.2458. The scale row has validation F1 0.8587, so the more local rule does not establish better generalisation.

### `SVC: kernel`

The same kernels are now compared on the athletes. The default RBF kernel, $\exp(-\gamma\lVert x-x'\rVert^2)$, bends the boundary through local similarities, while `linear` keeps a straight one. The symbols $r$ (`coef0`) and $d$ (`degree`) of `poly` and `sigmoid` are held at their defaults 0 and 3. Gamma applies to the three nonlinear kernels, but not linear. Kernel choice changes the similarity rule without changing the inherited features.

__Step 38:__ Compare `kernel` while keeping the other defaults. Fill the value list with `["linear", "rbf", "poly", "sigmoid"]` and measure every row on the shared folds.

In [ ]:
kernel_values = ...

__Step 39:__ Measure every kernel value on the prepared folds and fit each configuration on all training rows.

In [ ]:
kernel_rows = []
for value in tqdm(kernel_values, desc='kernel', leave=True, position=0):
    settings = {**fixed_settings, "kernel": value}
    train_scores, validation_scores, warning_counts = [], [], []
    for fold in prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = SVC(**settings).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
        warning_counts.append(sum(issubclass(w.category, ConvergenceWarning) for w in caught))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        fitted = SVC(**settings).fit(Z_train, y_train)
    kernel_rows.append({"name": f"kernel={value}", "settings": settings,
        "train": train_scores, "validation": validation_scores, "model": fitted,
        "configuration": " ".join(repr(SVC(**settings)).split()),
        "train F1": np.mean(train_scores), "cv F1": np.mean(validation_scores),
        "support vectors": int(fitted.n_support_.sum()), "iterations": int(fitted.n_iter_.max()),
        "fit status": int(fitted.fit_status_), "fold warnings": sum(warning_counts)})
kernel_table = pd.DataFrame(kernel_rows)

__Step 40:__ Display the kernel comparison with its held-out F1 and solver diagnostics.

In [ ]:
kernel_table["test F1"] = [f1_score(y_test, row["model"].predict(Z_test)) for row in kernel_rows]
display(kernel_table.drop(columns=["settings", "train", "validation", "model"]).round(4))


RBF has the largest validation F1, 0.8587. The four rows compare similarity rules with all other arguments at their defaults.

__Step 41:__ Record the validation-selected configuration.

In [ ]:
kernel_best = kernel_rows[int(kernel_table["cv F1"].idxmax())]
svm_log[:] = [row for row in svm_log if row["validation F1 is"] != 'one setting changed: kernel']
record(svm_log, {"frame": type(folds).__name__, "rows": len(y_train),
                 "folds": folds.get_n_splits(), "name": kernel_best["configuration"],
                 "train": kernel_best["train"], "validation": kernel_best["validation"],
                 "model": kernel_best["model"]},
       Z_test, y_test, 'one setting changed: kernel', metric=f1_score, label="F1")


__Step 42:__ Plot the measured comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
positions = np.arange(len(kernel_table))
for column, marker in [("train F1", "o"), ("cv F1", "s"), ("test F1", "^")]:
    ax.plot(positions, kernel_table[column], marker=marker, label=column)
ax.set(xticks=positions, xticklabels=kernel_table["name"].str.split("=").str[-1], xlabel='kernel', ylabel="F1",
       title="SVC: kernel compared on the shared folds")
ax.legend()
fig.tight_layout()
plt.show()


The largest validation F1 in this table is 0.8587, from `kernel=rbf`; its training F1 is 0.8874 and its held-out F1 is 0.8649. Linear gives validation F1 0.8532, while sigmoid gives 0.7306. Their scores compare different similarity rules under the same inherited features; the joint search still retains every kernel.

### `SVC: max_iter`

The iteration cap limits optimisation, not model capacity. A ConvergenceWarning or nonzero fit_status_ means the cap prevented the solver from satisfying its convergence criterion. A high F1 does not prove convergence; -1 allows the solver to run without a cap.

__Step 43:__ Compare `max_iter` while keeping the other defaults. Fill the value list with `[100, 1000, 10000, -1]` and measure every row on the shared folds.

In [ ]:
max_iter_values = ...

__Step 44:__ Measure every max_iter value on the prepared folds and fit each configuration on all training rows.

In [ ]:
max_iter_rows = []
for value in tqdm(max_iter_values, desc='max_iter', leave=True, position=0):
    settings = {**fixed_settings, "max_iter": value}
    train_scores, validation_scores, warning_counts = [], [], []
    for fold in prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = SVC(**settings).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
        warning_counts.append(sum(issubclass(w.category, ConvergenceWarning) for w in caught))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        fitted = SVC(**settings).fit(Z_train, y_train)
    max_iter_rows.append({"name": f"max_iter={value}", "settings": settings,
        "train": train_scores, "validation": validation_scores, "model": fitted,
        "configuration": " ".join(repr(SVC(**settings)).split()),
        "train F1": np.mean(train_scores), "cv F1": np.mean(validation_scores),
        "support vectors": int(fitted.n_support_.sum()), "iterations": int(fitted.n_iter_.max()),
        "fit status": int(fitted.fit_status_), "fold warnings": sum(warning_counts)})
max_iter_table = pd.DataFrame(max_iter_rows)

__Step 45:__ Display the max_iter comparison with its held-out F1 and solver diagnostics.

In [ ]:
max_iter_table["test F1"] = [f1_score(y_test, row["model"].predict(Z_test)) for row in max_iter_rows]
display(max_iter_table.drop(columns=["settings", "train", "validation", "model"]).round(4))


The 10000 cap and the unlimited fit tie on validation F1 at 0.8587, and the log records `10000` because `idxmax` returns the first of equal values. The `max_iter=1000` row also prints 0.8587, but it stopped at its cap in all ten fold fits (fold warnings 10), so only the two larger values finished optimisation. The table reports warnings and solver status, which assess optimisation separately.

__Step 46:__ Record the validation-selected configuration.

In [ ]:
max_iter_best = max_iter_rows[int(max_iter_table["cv F1"].idxmax())]
svm_log[:] = [row for row in svm_log if row["validation F1 is"] != 'one setting changed: max_iter']
record(svm_log, {"frame": type(folds).__name__, "rows": len(y_train),
                 "folds": folds.get_n_splits(), "name": max_iter_best["configuration"],
                 "train": max_iter_best["train"], "validation": max_iter_best["validation"],
                 "model": max_iter_best["model"]},
       Z_test, y_test, 'one setting changed: max_iter', metric=f1_score, label="F1")


__Step 47:__ Plot the measured comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
positions = np.arange(len(max_iter_table))
for column, marker in [("train F1", "o"), ("cv F1", "s"), ("test F1", "^")]:
    ax.plot(positions, max_iter_table[column], marker=marker, label=column)
ax.set(xticks=positions, xticklabels=max_iter_table["name"].str.split("=").str[-1], xlabel='max_iter', ylabel="F1",
       title="SVC: max_iter compared on the shared folds")
ax.legend()
fig.tight_layout()
plt.show()


The largest validation F1 in this table is 0.8587, from `max_iter=10000`; its training F1 is 0.8874 and its held-out F1 is 0.8649. A cap of 100 raises warnings in 10 of ten fold fits and returns full-training solver status 1. With no cap, the full fit takes 2285 iterations and returns status 0. Read predictive rank separately from completed optimisation.

### `SVC: class_weight`

Balanced weighting assigns each fitting class weight n/(k times n_c), where n is that fitting partition size, k the number of classes, and n_c its count in class c. These weights multiply C for that class. Reweighting can change the precision-recall balance; it does not directly optimise F1.

__Step 48:__ Compare `class_weight` while keeping the other defaults. Fill the value list with `[None, "balanced"]` and measure every row on the shared folds.

In [ ]:
class_weight_values = ...

__Step 49:__ Measure every class_weight value on the prepared folds and fit each configuration on all training rows.

In [ ]:
class_weight_rows = []
for value in tqdm(class_weight_values, desc='class_weight', leave=True, position=0):
    settings = {**fixed_settings, "class_weight": value}
    train_scores, validation_scores, warning_counts = [], [], []
    for fold in prepared:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = SVC(**settings).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
        warning_counts.append(sum(issubclass(w.category, ConvergenceWarning) for w in caught))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        fitted = SVC(**settings).fit(Z_train, y_train)
    class_weight_rows.append({"name": f"class_weight={value}", "settings": settings,
        "train": train_scores, "validation": validation_scores, "model": fitted,
        "configuration": " ".join(repr(SVC(**settings)).split()),
        "train F1": np.mean(train_scores), "cv F1": np.mean(validation_scores),
        "support vectors": int(fitted.n_support_.sum()), "iterations": int(fitted.n_iter_.max()),
        "fit status": int(fitted.fit_status_), "fold warnings": sum(warning_counts)})
class_weight_table = pd.DataFrame(class_weight_rows)

__Step 50:__ Display the class_weight comparison with its held-out F1 and solver diagnostics.

In [ ]:
class_weight_table["test F1"] = [f1_score(y_test, row["model"].predict(Z_test)) for row in class_weight_rows]
display(class_weight_table.drop(columns=["settings", "train", "validation", "model"]).round(4))


The unweighted row has validation F1 0.8587, compared with 0.8458 for balanced weighting.

__Step 51:__ Record the validation-selected configuration.

In [ ]:
class_weight_best = class_weight_rows[int(class_weight_table["cv F1"].idxmax())]
svm_log[:] = [row for row in svm_log if row["validation F1 is"] != 'one setting changed: class_weight']
record(svm_log, {"frame": type(folds).__name__, "rows": len(y_train),
                 "folds": folds.get_n_splits(), "name": class_weight_best["configuration"],
                 "train": class_weight_best["train"], "validation": class_weight_best["validation"],
                 "model": class_weight_best["model"]},
       Z_test, y_test, 'one setting changed: class_weight', metric=f1_score, label="F1")


__Step 52:__ Plot the measured comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
positions = np.arange(len(class_weight_table))
for column, marker in [("train F1", "o"), ("cv F1", "s"), ("test F1", "^")]:
    ax.plot(positions, class_weight_table[column], marker=marker, label=column)
ax.set(xticks=positions, xticklabels=class_weight_table["name"].str.split("=").str[-1], xlabel='class_weight', ylabel="F1",
       title="SVC: class_weight compared on the shared folds")
ax.legend()
fig.tight_layout()
plt.show()


The largest validation F1 in this table is 0.8587, from `class_weight=None`; its training F1 is 0.8874 and its held-out F1 is 0.8649. Balanced weighting changes validation F1 from 0.8587 to 0.8458 and held-out F1 from 0.8649 to 0.8551. The log follows validation; both held-out values remain descriptive.

# <font color='#E8800A'>The fixed search budget</font> <a class="anchor" id="budget"></a>
[Back to TOC](#toc)

RandomizedSearchCV draws sixty configurations from conditional distributions for SVC, NuSVC, LinearSVC and hinge-loss SGD. Positive parameters span orders of magnitude through log-uniform distributions. Kernel branches include gamma only when relevant, polynomial degree and nonnegative offset only for poly, and signed offsets for sigmoid. LinearSVC branches pair valid penalty, loss and dual settings; elastic-net SGD alone varies l1_ratio.

Each trial first chooses one distribution branch uniformly, then draws its parameters. Families with more branches receive more expected trials; there is no per-family quota. The fixed seed repeats the draws across the ten outer searches and final search. Counts come from fitted cv_results_, and every family remains eligible. Preprocessing learns within each fitting partition. Nu bounds use training labels only, including every planned inner fitting partition; held-out athletes never influence selection.


__Step 53:__ Declare valid conditional parameter distributions before observing search scores.

In [ ]:
inner_folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
nu_limits = []
for outer_fit in [np.arange(len(y_train))] + [fit for fit, _ in folds.split(X_train, y_train)]:
    outer_y = y_train.iloc[outer_fit]
    for inner_fit, _ in inner_folds.split(X_train.iloc[outer_fit], outer_y):
        counts = outer_y.iloc[inner_fit].value_counts()
        nu_limits.append(2 * counts.min() / counts.sum())
valid_nu = [0.05, 0.1, 0.25, 0.5, 0.7]
assert max(valid_nu) < min(nu_limits), "Revise nu bounds before evaluating the declared search"
svm_distributions = []
for estimator in [SVC(**fixed_settings), NuSVC(random_state=random_state, cache_size=256)]:
    common = {"estimator": [estimator], "estimator__tol": loguniform(1e-4, 1e-2),
              "estimator__shrinking": [False, True], "estimator__max_iter": [100, 1000, 10000, -1]}
    if isinstance(estimator, SVC):
        common.update({"estimator__C": loguniform(0.01, 100), "estimator__class_weight": [None, "balanced"]})
    else:
        common["estimator__nu"] = valid_nu
    for kernel in ["linear", "rbf", "poly", "sigmoid"]:
        branch = {**common, "estimator__kernel": [kernel]}
        if kernel != "linear":
            branch["estimator__gamma"] = loguniform(0.001, 10)
        if kernel == "poly":
            branch.update({"estimator__degree": [2, 3, 4, 5], "estimator__coef0": [0.0, 0.5, 1.0, 2.0]})
        if kernel == "sigmoid":
            branch["estimator__coef0"] = [-1.0, -0.5, 0.0, 0.5, 1.0]
        svm_distributions.append(branch)
for penalty, loss, dual in [("l2", "hinge", True), ("l2", "squared_hinge", True),
                             ("l2", "squared_hinge", False), ("l1", "squared_hinge", False)]:
    svm_distributions.append({"estimator": [LinearSVC(random_state=random_state)],
        "estimator__C": loguniform(0.01, 100), "estimator__penalty": [penalty],
        "estimator__loss": [loss], "estimator__dual": [dual],
        "estimator__class_weight": [None, "balanced"], "estimator__fit_intercept": [False, True],
        "estimator__intercept_scaling": loguniform(0.1, 10),
        "estimator__tol": loguniform(1e-5, 1e-2), "estimator__max_iter": [1000, 5000, 10000]})
for penalty in ["l2", "l1", "elasticnet"]:
    branch = {"estimator": [SGDClassifier(loss="hinge", random_state=random_state)],
        "estimator__alpha": loguniform(1e-5, 1e-2), "estimator__penalty": [penalty],
        "estimator__class_weight": [None, "balanced"], "estimator__average": [False, True],
        "estimator__fit_intercept": [False, True], "estimator__max_iter": [1000, 5000, 10000],
        "estimator__tol": loguniform(1e-5, 1e-2), "estimator__learning_rate": ["optimal", "adaptive"],
        "estimator__eta0": loguniform(1e-3, 0.1)}
    if penalty == "elasticnet":
        branch["estimator__l1_ratio"] = [0.05, 0.15, 0.5, 0.85, 0.95]
    svm_distributions.append(branch)


__Step 54:__ Inspect the trial budget and feasible nu values.

In [ ]:
print("Conditional distribution branches:", len(svm_distributions))
print("Random trials per search:", 60)
print("Feasible nu values:", valid_nu, "minimum inner-partition bound:", min(nu_limits))


# <font color='#E8800A'>Nested cross-validation</font> <a class="anchor" id="nested"></a>
[Back to TOC](#toc)

Each outer training partition runs the complete sixty-candidate, four-family search with three shuffled stratified inner folds. Preprocessing learns only from each inner fitting partition. The outer validation athletes are excluded from candidate ranking, so the ten scores assess the complete family-and-settings selection procedure.

Capped kernel fits and iterative linear solvers can raise convergence warnings. The search cells suppress that warning category, while the family reference table and final diagnostics make optimisation limits visible. The untuned SVC uses the same outer splits. These repeated folds overlap, so their differences and standard deviation are descriptive.

__Step 55:__ Construct the seeded sixty-trial random search.

In [ ]:
base_svm = PreparedEstimator(FoldPreprocessor(classification_recipe), SVC(**fixed_settings))
svm_template = CachedSearchCV(RandomizedSearchCV(base_svm, svm_distributions,
    n_iter=60, random_state=33, scoring="f1", cv=inner_folds, n_jobs=2, error_score="raise"))


__Step 56:__ Run the nested search on each outer training fold.

In [ ]:
with parallel_config(backend="loky", initializer=ignore_svc_search_warnings), warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    svm_nested_fold_results = [cross_validate(svm_template, X_train, y_train, cv=[_outer_split], scoring='f1', return_train_score=True, n_jobs=1)
        for _outer_split in tqdm(list(folds.split(X_train, y_train)), total=folds.get_n_splits(X_train, y_train), desc='Nested CV: outer folds', unit='fold', leave=True, position=0)]
    svm_nested = {key: np.concatenate([np.asarray(result[key], dtype=object if key == 'estimator' else None) for result in svm_nested_fold_results]) for key in svm_nested_fold_results[0]}
paired_gain = svm_nested["test_score"] - np.asarray(reference["validation"])
print(f"reference CV F1: {np.mean(reference['validation']):.4f}")
print(f"nested F1: {svm_nested['test_score'].mean():.4f}")
print(f"outer-fold SD: {svm_nested['test_score'].std(ddof=1):.4f}")
print(f"paired gain: {paired_gain.mean():+.4f}")

The sixty-trial search averages 0.8510 outer-fold F1, against 0.8587 for the untuned SVC on the same splits. Its paired mean difference is -0.0077, and it scores higher in 1 of ten folds. The descriptive outer-fold sample standard deviation is 0.0157. These overlapping folds evaluate the complete family-and-settings selection procedure.

# <font color='#E8800A'>The final search and its winner</font> <a class="anchor" id="search"></a>
[Back to TOC](#toc)

The final search fits the same sixty seeded random draws on all training athletes and selects the highest mean inner F1.

__Step 57:__ Fit the final search and inspect its selected settings.

In [ ]:
with parallel_config(backend="loky", initializer=ignore_svc_search_warnings), warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    svm_search = clone(svm_template).fit(X_train, y_train)
svm_candidates = svm_search.cv_results_["params"]
candidate_families = [type(candidate["estimator"]).__name__ for candidate in svm_candidates]
display(pd.Series(candidate_families).value_counts().rename("actual sampled trials"))
ranking = pd.DataFrame({"settings": svm_search.cv_results_["params"],
                        "family": candidate_families, "inner F1": svm_search.cv_results_["mean_test_score"]})

print(f"best inner F1: {svm_search.best_score_:.4f}")
print("best parameters:", svm_search.best_params_)
winner_model = svm_search.best_estimator_
winner_constructor = clone(winner_model.estimator_)
print(winner_constructor)


__Step 58:__ Inspect the selected family and its fitted representation.

In [ ]:
selected_estimator = winner_model.estimator_
print("Selected family:", type(selected_estimator).__name__)
if hasattr(selected_estimator, "n_support_"):
    print("Support vectors by class:", selected_estimator.n_support_)
else:
    print("Linear coefficient matrix:", selected_estimator.coef_.shape)


The selected LinearSVC stores a coefficient matrix of shape [1, 28] and its fitted intercept. Its decision function uses this fitted representation of the prepared athlete features.

__Step 59:__ Read the selected solver diagnostics.

In [ ]:
if hasattr(selected_estimator, "fit_status_"):
    print("Solver status:", selected_estimator.fit_status_)
print("Iterations:", selected_estimator.n_iter_)
print("Iteration cap:", selected_estimator.max_iter)


The selected LinearSVC reports 10000 solver iterations with max_iter=10000. Read these diagnostics alongside F1: predictive performance does not establish completed optimisation.

__Step 60:__ Read the five candidates with the highest inner scores.

In [ ]:
top_five = ranking.nlargest(5, "inner F1").copy()
top_five_test = []
for candidate in top_five["settings"]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        diagnostic_model = clone(base_svm).set_params(**clone(candidate, safe=False)).fit(X_train, y_train)
    top_five_test.append(f1_score(y_test, diagnostic_model.predict(X_test)))
top_five["test F1 (descriptive)"] = top_five_test
display(top_five)


The top-five inner ranking contains SGDClassifier: 3, LinearSVC: 2. Its leading mean inner F1 is 0.8537. The inner score selects the first row; held-out F1 remains descriptive and never changes the ranking.

__Step 61:__ Measure the chosen configuration on the prepared folds.

In [ ]:
winner_train, winner_validation = [], []
for fold in prepared:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        winner_fold = clone(winner_constructor).fit(fold["X_fit"], fold["y_fit"])
    winner_train.append(f1_score(fold["y_fit"], winner_fold.predict(fold["X_fit"])))
    winner_validation.append(f1_score(fold["y_score"], winner_fold.predict(fold["X_score"])))
winner_train = np.array(winner_train)
winner_validation = np.array(winner_validation)
winner_name = " ".join(repr(winner_constructor).split())


__Step 62:__ Record the model and its fold scores.

In [ ]:
record(svm_log, {"frame": type(folds).__name__, "rows": len(y_train),
                 "folds": folds.get_n_splits(), "name": winner_name,
                 "train": winner_train, "validation": winner_validation, "model": winner_model},
       X_test, y_test, "winner's own score", metric=f1_score, label="F1")


__Step 63:__ Record the model and its fold scores.

In [ ]:
record(svm_log, {"frame": type(folds).__name__, "rows": len(y_train),
                 "folds": folds.get_n_splits(), "name": f"joint search over {len(svm_candidates)} sampled combinations",
                 "train": svm_nested["train_score"], "validation": svm_nested["test_score"], "model": winner_model},
       X_test, y_test, "nested estimate", metric=f1_score, label="F1")

The winner's own shared-fold score is measured after selecting its family and settings, so it does not replace the nested estimate. Both log entries describe the same fitted classifier on the holdout.

For SVC and NuSVC the score is $f(x)=\sum_j a_jK(s_j,x)+b$; for LinearSVC and hinge-loss SGD it is $f(x)=w^\top x+b$. `decision_function` returns the fitted score and `predict` returns its class. A positive binary score selects the second entry in `classes_`; the score is not a probability.

__Step 64:__ Inspect the selected classifier on held-out athletes.

In [ ]:
predictions = winner_model.predict(X_test)
decisions = winner_model.decision_function(X_test)
fitted_svm = winner_model.estimator_
prediction_table = pd.DataFrame({"actual": y_test.iloc[:5], "prediction": predictions[:5],
                                "decision score": decisions[:5]}, index=y_test.index[:5])
display(prediction_table)
print("class order:", fitted_svm.classes_)
print(f"held-out F1: {f1_score(y_test, predictions):.4f}")

The selected LinearSVC has its own shared-fold F1 of 0.8537 and held-out F1 of 0.8524. The board pairs the nested procedure estimate, 0.8510, with this fitted classifierÃ¢â‚¬â„¢s held-out result.

# <font color='#E8800A'>The course board</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The board publishes the nested estimate of the fixed search and its final winner's held-out F1. Earlier families enter through compatible saved rows, including their saved baselines. No earlier model or baseline is refitted.

__Step 65:__ Display the accumulated model attempts.

In [ ]:
show_log(svm_log, label="F1")


__Step 66:__ Record the model and its fold scores.

In [ ]:
results_dir = "../../results"
floor = f1_score(y_test, np.full(len(y_test), y_train.mode().iat[0]))
board = Scoreboard("champions outcome", "F1", floor=floor, context=result_context(
    "../../data/interim/champions.csv", classification_recipe_log,
    y_train.index.tolist(), y_test.index.tolist(), outer_folds=outer_folds(folds, X_train, y_train)))
for week, model, cv, test in results_before(14, "classification", results_dir,
                                          metric=board.metric, context=board.context):
    board.record(week, model, cv, test)
board.record(14, winner_name,
             svm_nested["test_score"].mean(), f1_score(y_test, predictions))
board.save(results_dir, 14, "classification", baseline_cv=np.mean(reference["validation"]),
           baseline_test=f1_score(y_test, reference["model"].predict(X_test)))
print(board.table())
joblib.dump(winner_model, Path(results_dir) / "week_14_classification_model.joblib")

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

A supporting row contributes to the fitted boundary, while C changes the cost of its violations. Gamma changes nonlinear similarities and does nothing for a linear kernel. Optimisation status and validation F1 answer different questions. The SVC sweeps describe individual effects; the seeded random search selects a family and its settings. Inner scores select, outer scores evaluate the selection procedure, and held-out athletes describe the final model.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- Bergstra, J. and Bengio, Y. (2012). [Random Search for Hyper-Parameter Optimization](https://www.jmlr.org/papers/v13/bergstra12a.html), JMLR 13, 281-305.
- scikit-learn [SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) and [support vector machines](https://scikit-learn.org/stable/modules/svm.html).
- scikit-learn [ParameterSampler](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ParameterSampler.html): distributions sample with replacement, using the fixed random seed.
- scikit-learn [nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).